# Gestionnaire de tâches avec ipywidgets

Ce TP construit un gestionnaire de tâches dans le notebook : une saisie et un bouton pour ajouter une tâche, la liste des tâches, des boutons pour les marquer faites ou les supprimer, et un filtre (toutes, à faire, faites). Il suit l'organisation Modèle-Vue-Contrôleur du cours : le modèle dans un fichier, `modele.py`, la vue et le contrôleur dans les cellules du notebook, avec ipywidgets. Les widgets s'affichent sous les cellules, dans Colab comme dans Jupyter ou VS Code (le Codespace).

C'est le jumeau du TP Tkinter, avec le même modèle : si vous l'avez déjà écrit, reprenez votre `modele.py`.

C'est un TP linéaire : chaque partie s'appuie sur la précédente. Si vous bloquez, exécutez la solution et passez à la suite.

## 1. Le modèle

Écrivez `modele.py`, du Python pur, sans interface : c'est lui qui porte les règles de l'application.

- `FILTRES = ("toutes", "à faire", "faites")`, les filtres que l'interface proposera.
- `Tache`, une dataclass : un `titre` (`str`) et `faite` (`bool`, `False` par défaut). Le titre perd ses espaces de début et de fin, et un titre vide lève une `ValueError`.
- `ListeTaches`, une dataclass dont l'unique attribut, `taches`, est une liste de `Tache`, vide au départ, avec quatre méthodes :
  - `ajouter(titre)` crée la tâche, l'ajoute à la fin de la liste et la renvoie ; un titre déjà présent lève une `ValueError` ;
  - `supprimer(tache)` la retire de la liste ;
  - `basculer(tache)` la marque faite si elle était à faire, et inversement ;
  - `filtrer(filtre)` renvoie la liste des tâches qui correspondent à l'un des `FILTRES`, dans l'ordre d'ajout ; un autre filtre lève une `ValueError`.

Les messages des `ValueError` s'adressent à l'utilisateur : l'interface les affichera tels quels.

La cellule commence par `%%writefile modele.py` : Jupyter écrit le reste de la cellule dans ce fichier au lieu de l'exécuter. La cellule de vérification, après la solution, teste votre modèle.

**Documentation utile**

- [%%writefile](https://ipython.readthedocs.io/en/stable/interactive/magics.html#cellmagic-writefile)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Le cours sur les classes de données montre les deux outils dont `Tache` et `ListeTaches` ont besoin : `__post_init__()`, que la dataclass appelle après son `__init__` généré, valide le titre (`self.titre = self.titre.strip()`, puis `if not self.titre: ...`) ; `field(default_factory=list)` donne à chaque `ListeTaches` sa propre liste. `any(...)` sur un générateur dit si une tâche a déjà ce titre ; comparez au titre nettoyé, `"  Pain "` est un doublon de `"Pain"`. `basculer` n'a qu'une ligne : `tache.faite = not tache.faite`.

```python
%%writefile modele.py
from dataclasses import dataclass, field

FILTRES = ("toutes", "à faire", "faites")


@dataclass
class Tache:
  titre: str
  faite: bool = False

  def __post_init__(self) -> None:
    self.titre = self.titre.strip()
    if not self.titre:
      msg = "une tâche a besoin d'un titre"
      raise ValueError(msg)


@dataclass
class ListeTaches:
  taches: list[Tache] = field(default_factory=list)

  def ajouter(self, titre: str) -> Tache:
    tache = Tache(titre)
    if any(...):
      msg = f"« {tache.titre} » est déjà dans la liste"
      raise ValueError(msg)
    ...

  def supprimer(self, tache: Tache) -> None: ...

  def basculer(self, tache: Tache) -> None: ...

  def filtrer(self, filtre: str) -> list[Tache]:
    if filtre == "toutes":
      return list(self.taches)
    ...
```

### Solution

In [ ]:
%%writefile modele.py
from dataclasses import dataclass, field

FILTRES = ("toutes", "à faire", "faites")


@dataclass
class Tache:
  titre: str
  faite: bool = False

  def __post_init__(self) -> None:
    self.titre = self.titre.strip()
    if not self.titre:
      msg = "une tâche a besoin d'un titre"
      raise ValueError(msg)


@dataclass
class ListeTaches:
  taches: list[Tache] = field(default_factory=list)

  def ajouter(self, titre: str) -> Tache:
    tache = Tache(titre)
    if any(autre.titre == tache.titre for autre in self.taches):
      msg = f"« {tache.titre} » est déjà dans la liste"
      raise ValueError(msg)
    self.taches.append(tache)
    return tache

  def supprimer(self, tache: Tache) -> None:
    self.taches.remove(tache)

  def basculer(self, tache: Tache) -> None:
    tache.faite = not tache.faite

  def filtrer(self, filtre: str) -> list[Tache]:
    if filtre == "toutes":
      return list(self.taches)
    if filtre == "à faire":
      return [tache for tache in self.taches if not tache.faite]
    if filtre == "faites":
      return [tache for tache in self.taches if tache.faite]
    msg = f"filtre inconnu : {filtre!r}"
    raise ValueError(msg)

### Vérification

La cellule suivante recharge `modele.py` (le notebook garde sinon la version qu'il a importée en premier), puis joue un petit scénario. Résultats attendus : `['Acheter du pain']` à faire, `['Réviser le cours']` faite, puis trois `ValueError` (titre vide, doublon, filtre inconnu), et enfin `['Acheter du pain']` après la suppression.

In [ ]:
import importlib

import modele

importlib.reload(modele)

liste = modele.ListeTaches()
liste.ajouter("Acheter du pain")
revision = liste.ajouter("  Réviser le cours ")
liste.basculer(revision)
print("À faire :", [tache.titre for tache in liste.filtrer("à faire")])
print("Faites :", [tache.titre for tache in liste.filtrer("faites")])

for essai in (
  lambda: liste.ajouter("   "),
  lambda: liste.ajouter("Acheter du pain"),
  lambda: liste.filtrer("urgentes"),
):
  try:
    essai()
  except ValueError as erreur:
    print("ValueError :", erreur)
  else:
    print("Pas de ValueError : à revoir")

liste.supprimer(revision)
print("Toutes :", [tache.titre for tache in liste.filtrer("toutes")])

## 2. La vue

Écrivez une classe `VueTaches`, comme la vue ipywidgets du cours. Elle ne connaît pas le modèle : elle n'importe de `modele.py` que `FILTRES`. Elle crée ces widgets, dans ses attributs :

- `saisie`, un `Text`, et le bouton `ajouter` (« Ajouter »), sur une ligne ;
- `liste`, un `SelectMultiple` de 10 lignes, vide au départ ;
- sur une ligne, les boutons `basculer` (« Faite / à faire ») et `supprimer` (« Supprimer »), et `filtre`, un `Dropdown` des `FILTRES` ;
- `etat`, un `Label`, et `message`, un `HTML` pour les erreurs.

Elle les range dans `racine`, une `VBox`, que le notebook affichera. Le contrôleur se servira de ces noms d'attributs : gardez-les.

La vue a trois méthodes, que le contrôleur appellera :

- `afficher(lignes, etat)` remplace le contenu de la liste par `lignes`, une liste de chaînes, le texte d'`etat` par `etat`, et efface le message ;
- `selection()` renvoie la liste des indices des lignes sélectionnées ;
- `signaler(message)` affiche le message en gras et en rouge dans `message`.

La cellule qui suit la solution affiche un aperçu de la vue, avec deux lignes d'exemple : les boutons n'y font encore rien.

**Documentation utile**

- [SelectMultiple](https://ipywidgets.readthedocs.io/en/stable/examples/Widget%20List.html#selectmultiple)
- [HTML](https://ipywidgets.readthedocs.io/en/stable/examples/Widget%20List.html#html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `widgets.SelectMultiple(options=[], rows=10)` : `rows` fixe sa hauteur, en lignes. Ctrl+clic et Maj+clic sélectionnent plusieurs lignes.
- `widgets.Dropdown(options=FILTRES)` sélectionne la première option au départ.
- `racine` est une `VBox` de `HBox` et de widgets : `widgets.VBox([widgets.HBox([self.saisie, self.ajouter]), self.liste, ...])`.
- Une liste se remplace en entier : `self.liste.options = lignes`. Son `index` est le tuple des indices sélectionnés.
- Un `HTML` affiche du HTML : `f"<b style='color: crimson'>{message}</b>"`.

```python
import ipywidgets as widgets
from IPython.display import display

from modele import FILTRES


class VueTaches:
  def __init__(self) -> None:
    self.saisie = widgets.Text(placeholder="Nouvelle tâche")
    self.ajouter = widgets.Button(description="Ajouter")
    self.liste = ...
    self.basculer = widgets.Button(description="Faite / à faire")
    ...
    self.etat = widgets.Label()
    self.message = widgets.HTML()
    self.racine = widgets.VBox([widgets.HBox([self.saisie, self.ajouter]), ...])

  def afficher(self, lignes: list[str], etat: str) -> None:
    self.liste.options = lignes
    ...

  def selection(self) -> list[int]:
    return ...

  def signaler(self, message: str) -> None: ...
```

### Solution

In [ ]:
import ipywidgets as widgets
from IPython.display import display

from modele import FILTRES


class VueTaches:
  def __init__(self) -> None:
    self.saisie = widgets.Text(placeholder="Nouvelle tâche")
    self.ajouter = widgets.Button(description="Ajouter")
    self.liste = widgets.SelectMultiple(options=[], rows=10)
    self.basculer = widgets.Button(description="Faite / à faire")
    self.supprimer = widgets.Button(description="Supprimer")
    self.filtre = widgets.Dropdown(options=FILTRES)
    self.etat = widgets.Label()
    self.message = widgets.HTML()
    self.racine = widgets.VBox(
      [
        widgets.HBox([self.saisie, self.ajouter]),
        self.liste,
        widgets.HBox([self.basculer, self.supprimer, self.filtre]),
        self.etat,
        self.message,
      ]
    )

  def afficher(self, lignes: list[str], etat: str) -> None:
    self.liste.options = lignes
    self.etat.value = etat
    self.message.value = ""

  def selection(self) -> list[int]:
    return list(self.liste.index)

  def signaler(self, message: str) -> None:
    self.message.value = f"<b style='color: crimson'>{message}</b>"

### Aperçu

In [ ]:
apercu = VueTaches()
apercu.afficher(["[ ] Acheter du pain", "[x] Réviser le cours"], "1 tâche(s) à faire")
apercu.signaler("Exemple d'erreur")
display(apercu.racine)

## 3. Le contrôleur

Écrivez une classe `Controleur` qui reçoit le modèle et la vue, branche les widgets sur ses méthodes et tient la vue à jour :

- `rafraichir()` demande au modèle les tâches du filtre choisi, les garde dans `self.visibles`, et les affiche, une ligne par tâche, sous la forme `[ ] titre` (à faire) ou `[x] titre` (faite), avec le nombre de tâches à faire dans `etat`. La ligne `i` de la liste est la tâche `self.visibles[i]`.
- `ajouter()` ajoute la tâche saisie. Si le modèle lève une `ValueError`, la vue signale son message ; sinon, la saisie se vide et la liste se rafraîchit. Le bouton « Ajouter » l'appelle.
- `basculer()` et `supprimer()` agissent sur les tâches sélectionnées, puis rafraîchissent la liste.
- Choisir un filtre rafraîchit la liste.

La cellule qui suit la solution crée l'application et l'affiche.

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `on_click` passe le bouton cliqué au callback, dont on n'a pas besoin ici : `vue.ajouter.on_click(lambda bouton: self.ajouter())`.
- `observe` lui passe un dict qui décrit le changement : `vue.filtre.observe(lambda change: self.rafraichir(), names="value")`.
- `try` / `except ValueError as erreur` / `else`, comme dans le contrôleur du cours : le `else` ne s'exécute que si l'ajout a réussi.
- La liste n'affiche que les tâches du filtre courant : sa ligne `i` n'est pas `modele.taches[i]`, d'où `self.visibles`. Une méthode `selectionnees()` qui renvoie `[self.visibles[i] for i in self.vue.selection()]` sert à `basculer` et à `supprimer`.
- C'est le contrôleur du TP Tkinter, aux branchements près : `on_click` et `observe` au lieu de `command=` et `bind`, et `value` au lieu de `get()` et `set()`.

```python
from modele import ListeTaches, Tache


class Controleur:
  def __init__(self, modele: ListeTaches, vue: VueTaches) -> None:
    self.modele = modele
    self.vue = vue
    self.visibles: list[Tache] = []
    vue.ajouter.on_click(lambda bouton: self.ajouter())
    ...
    self.rafraichir()

  def rafraichir(self) -> None:
    self.visibles = self.modele.filtrer(self.vue.filtre.value)
    lignes = []
    for tache in self.visibles:
      ...
    restantes = ...
    self.vue.afficher(lignes, f"{restantes} tâche(s) à faire")

  def ajouter(self) -> None:
    try:
      ...
    except ValueError as erreur:
      ...
    else:
      ...

  def selectionnees(self) -> list[Tache]:
    return [self.visibles[index] for index in self.vue.selection()]

  def basculer(self) -> None: ...

  def supprimer(self) -> None: ...
```

### Solution

In [ ]:
from modele import ListeTaches, Tache


class Controleur:
  def __init__(self, modele: ListeTaches, vue: VueTaches) -> None:
    self.modele = modele
    self.vue = vue
    self.visibles: list[Tache] = []
    vue.ajouter.on_click(lambda bouton: self.ajouter())
    vue.basculer.on_click(lambda bouton: self.basculer())
    vue.supprimer.on_click(lambda bouton: self.supprimer())
    vue.filtre.observe(lambda change: self.rafraichir(), names="value")
    self.rafraichir()

  def rafraichir(self) -> None:
    self.visibles = self.modele.filtrer(self.vue.filtre.value)
    lignes = []
    for tache in self.visibles:
      case = "[x]" if tache.faite else "[ ]"
      lignes.append(f"{case} {tache.titre}")
    restantes = len(self.modele.filtrer("à faire"))
    self.vue.afficher(lignes, f"{restantes} tâche(s) à faire")

  def ajouter(self) -> None:
    try:
      self.modele.ajouter(self.vue.saisie.value)
    except ValueError as erreur:
      self.vue.signaler(str(erreur))
    else:
      self.vue.saisie.value = ""
      self.rafraichir()

  def selectionnees(self) -> list[Tache]:
    return [self.visibles[index] for index in self.vue.selection()]

  def basculer(self) -> None:
    for tache in self.selectionnees():
      self.modele.basculer(tache)
    self.rafraichir()

  def supprimer(self) -> None:
    for tache in self.selectionnees():
      self.modele.supprimer(tache)
    self.rafraichir()

### L'application

Essayez : ajoutez des tâches, puis un titre vide et un doublon ; basculez plusieurs tâches d'un coup (Ctrl+clic) ; changez de filtre ; supprimez.

In [ ]:
vue = VueTaches()
controleur = Controleur(ListeTaches(), vue)
display(vue.racine)

### Vérification sans souris

La cellule suivante clique à votre place, avec la méthode `click()` des boutons, sur l'application affichée plus haut. Résultats attendus : deux tâches à faire et le message du doublon, puis `[x] Réviser le cours` faite, seule dans le filtre `faites`, et enfin `[ ] Acheter du pain` seule après sa suppression.

In [ ]:
vue.saisie.value = "Acheter du pain"
vue.ajouter.click()
vue.saisie.value = "Réviser le cours"
vue.ajouter.click()
vue.saisie.value = "Acheter du pain"
vue.ajouter.click()
print(vue.liste.options, vue.etat.value, vue.message.value)

vue.liste.value = ("[ ] Réviser le cours",)
vue.basculer.click()
print(vue.liste.options, vue.etat.value)

vue.filtre.value = "faites"
print(vue.liste.options)
vue.liste.value = vue.liste.options
vue.supprimer.click()
vue.filtre.value = "toutes"
print(vue.liste.options, vue.etat.value)